# Note 01 deep dive: the information layers of language in code

Companion notebook for `01-Information-Layers-of-SNLP.md`. Every number quoted in the note's worked examples comes from this notebook.

| Part | Layer | What it does |
|---|---|---|
| A | Phonology | Look up phonemes in the CMU Pronouncing Dictionary; implement the English plural and past-tense rules and test them against the dictionary |
| B | Morphology | Hand segmentation of English, Kannada, Turkish and Hindi words; a tiny greedy suffix stripper |
| C | Syntax | A CKY parser written from scratch, the full chart for a short sentence, parse counting, and a comparison with `nltk.ChartParser` |
| D | Syntax (ambiguity) | How the number of parses grows with the number of PPs (Catalan numbers) |
| E | Semantics | WordNet relations: synonymy, antonymy, hyponymy/hypernymy, meronymy, path similarity |
| F | Semantics (WSD) | Simplified Lesk by hand vs `nltk.wsd.lesk` |
| G | Semantics (logic) | First-order logic with `nltk.sem`: parse formulas and evaluate them in a small model |
| H | All layers | An ambiguity tour: lexical, POS, structural |
| I | Phonetics | Uniform-tube formants; source–filter vowel synthesis; LPC formant estimation |
| J | Phonology | Minimal pairs mined from CMUdict; a general A → B / C _ D rule applier (nasal assimilation) |
| K | Pragmatics & discourse | A rule-based speech-act tagger; a recency + agreement pronoun resolver; centering transitions |
| L | Practice-problem checks | Every computable answer in the note's practice bank |

Only NLTK, NumPy and SciPy are needed (no GPU). Data packages are downloaded quietly in the first cell.

In [1]:
import itertools, math, re
from collections import defaultdict, Counter
import nltk
for p in ["cmudict", "wordnet", "omw-1.4", "punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)
from nltk.corpus import cmudict, wordnet as wn, stopwords
print("NLTK", nltk.__version__)

NLTK 3.10.0


## Part A — Phonology: phonemes from the CMU dictionary and the plural rule

The CMU Pronouncing Dictionary (about 134k English words) writes pronunciations in **ARPAbet**, an ASCII version of the IPA for American English. Vowels carry a stress digit (0 = unstressed, 1 = primary, 2 = secondary).

In [2]:
cmu = cmudict.dict()
print("entries:", len(cmu))
for w in ["cat", "bat", "cats", "dogs", "horses", "walked", "played", "wanted", "though", "through", "tough"]:
    print(f"{w:8s}", cmu[w][0])

entries: 123455
cat      ['K', 'AE1', 'T']
bat      ['B', 'AE1', 'T']
cats     ['K', 'AE1', 'T', 'S']
dogs     ['D', 'AA1', 'G', 'Z']
horses   ['HH', 'AO1', 'R', 'S', 'AH0', 'Z']
walked   ['W', 'AO1', 'K', 'T']
played   ['P', 'L', 'EY1', 'D']
wanted   ['W', 'AA1', 'N', 'T', 'AH0', 'D']
though   ['DH', 'OW1']
through  ['TH', 'R', 'UW1']
tough    ['T', 'AH1', 'F']


### The rule, written as a phonological rewrite rule

Underlying plural morpheme /z/. Two ordered rules:

1. **Epenthesis:** insert a vowel between two sibilants: ∅ → ɪ / [+sibilant] _ z
2. **Voicing assimilation:** z → s / [−voice] _

The same machinery gives the past tense with underlying /d/: insert ɪ after /t/ or /d/; devoice to /t/ after a voiceless sound.

In [3]:
SIBILANTS = {"S", "Z", "SH", "ZH", "CH", "JH"}
VOICELESS = {"P", "T", "K", "F", "TH", "S", "SH", "CH", "HH"}

def strip(ph):                       # drop stress digits
    return [re.sub(r"\d", "", p) for p in ph]

def plural_suffix(singular_phones):
    last = strip(singular_phones)[-1]
    if last in SIBILANTS: return ["IH", "Z"]      # rule 1: epenthesis
    if last in VOICELESS: return ["S"]            # rule 2: devoicing
    return ["Z"]                                  # elsewhere: voiced /z/

def past_suffix(stem_phones):
    last = strip(stem_phones)[-1]
    if last in {"T", "D"}: return ["IH", "D"]
    if last in VOICELESS: return ["T"]
    return ["D"]

for w in ["cat", "dog", "horse", "bush", "judge", "cliff", "bee", "book"]:
    print(f"{w:6s} {' '.join(strip(cmu[w][0])):14s} + PL -> {'/'+''.join(plural_suffix(cmu[w][0]))+'/'}")

cat    K AE T         + PL -> /S/
dog    D AO G         + PL -> /Z/
horse  HH AO R S      + PL -> /IHZ/
bush   B UH SH        + PL -> /IHZ/
judge  JH AH JH       + PL -> /IHZ/
cliff  K L IH F       + PL -> /S/
bee    B IY           + PL -> /Z/
book   B UH K         + PL -> /S/


Now test the rule on every regular noun pair in a list against the dictionary's own pronunciation of the plural form.

In [4]:
nouns = ["cat","dog","horse","bus","book","bag","cup","pen","bridge","church","dish","rose","map","bed",
         "tree","shoe","cliff","wave","month","bath","garage","box","quiz","lamp","road","sock","card","fax",
         "class","watch","judge","page","face","toy","day","key","hat","kid","rock","song","myth","clothe"]
past_pairs = [("walk","walked"),("play","played"),("want","wanted"),("need","needed"),("jump","jumped"),
              ("love","loved"),("kiss","kissed"),("wish","wished"),("rob","robbed"),("hug","hugged"),
              ("start","started"),("end","ended"),("laugh","laughed"),("fix","fixed"),("rain","rained"),("bathe","bathed")]

def check(pairs, suffix_fn):
    ok, bad = 0, []
    for base, form in pairs:
        if base not in cmu or form not in cmu: continue
        pred = strip(cmu[base][0]) + suffix_fn(cmu[base][0])
        golds = [strip(p) for p in cmu[form]]
        # vowel quality of the inserted schwa varies (AH vs IH) in CMU: treat AH0/IH0 as the same reduced vowel
        norm = lambda ph: ["IH" if x == "AH" else x for x in ph]
        if any(norm(pred) == norm(g) for g in golds): ok += 1
        else: bad.append((base, form, " ".join(pred), [" ".join(g) for g in golds]))
    return ok, bad

pl_pairs = [(n, n + ("es" if re.search(r"(s|x|z|ch|sh)$", n) else "s")) for n in nouns]
ok, bad = check(pl_pairs, plural_suffix)
print(f"plural rule: {ok}/{ok+len(bad)} correct")
for b in bad: print("  mismatch:", b)
ok, bad = check(past_pairs, past_suffix)
print(f"past-tense rule: {ok}/{ok+len(bad)} correct")
for b in bad: print("  mismatch:", b)

plural rule: 41/41 correct
past-tense rule: 16/16 correct


Any mismatches are informative: e.g. *bath → baths* can be pronounced with /θs/ or /ðz/ (the stem's final consonant itself voices, a lexical exception), and the dictionary often lists several variants. A two-line rule covers almost all regular nouns and verbs — which is why rule-based phonology still powers the letter-to-sound component of TTS front ends.

In [5]:
# Distribution of plural allomorphs over all CMU words that end in a plain 's' whose stem is also in the dictionary
counts = Counter()
for w, prons in cmu.items():
    if w.endswith("s") and not w.endswith("ss") and w[:-1] in cmu and len(w) > 3:
        last = strip(prons[0])[-1]
        tail = strip(prons[0])[-2:]
        if tail == ["IH", "Z"] or tail == ["AH", "Z"]: counts["/ɪz/"] += 1
        elif last == "S": counts["/s/"] += 1
        elif last == "Z": counts["/z/"] += 1
print(counts, "total", sum(counts.values()))

Counter({'/z/': 8384, '/s/': 3755, '/ɪz/': 1075}) total 13214


## Part B — Morphology: segmentation across language types

Hand segmentations (gloss conventions as in the Leipzig glossing rules: `-` separates morphemes, capitals are grammatical morphemes).

In [6]:
segmentations = [
    ("unhappiness",        "English",  ["un", "happy", "ness"],             "NEG-happy-NMLZ"),
    ("internationalization","English", ["inter", "nation", "al", "ize", "ation"], "between-nation-ADJ-VERB-NMLZ"),
    ("manegaḷalli",        "Kannada",  ["mane", "gaḷ", "alli"],             "house-PL-LOC  'in the houses'"),
    ("maragaḷannu",        "Kannada",  ["mara", "gaḷ", "annu"],             "tree-PL-ACC   'the trees (object)'"),
    ("vīṭukaḷil",          "Tamil",    ["vīṭu", "kaḷ", "il"],               "house-PL-LOC  'in the houses' (cf. singular vīṭṭil: oblique stem vīṭṭ-)"),
    ("evlerimizden",       "Turkish",  ["ev", "ler", "imiz", "den"],        "house-PL-1PL.POSS-ABL 'from our houses'"),
    ("laṛkiyõ",            "Hindi",    ["laṛk", "iyõ"],                     "child-F.PL.OBL (one suffix = gender+number+case)"),
    ("went",               "English",  ["go+PAST"],                         "suppletion: no segmentable boundary"),
]
for w, lang, ms, gloss in segmentations:
    print(f"{lang:8s} {w:22s} {'-'.join(ms):28s} {len(ms)} morph  {gloss}")

English  unhappiness            un-happy-ness                3 morph  NEG-happy-NMLZ
English  internationalization   inter-nation-al-ize-ation    5 morph  between-nation-ADJ-VERB-NMLZ
Kannada  manegaḷalli            mane-gaḷ-alli                3 morph  house-PL-LOC  'in the houses'
Kannada  maragaḷannu            mara-gaḷ-annu                3 morph  tree-PL-ACC   'the trees (object)'
Tamil    vīṭukaḷil              vīṭu-kaḷ-il                  3 morph  house-PL-LOC  'in the houses' (cf. singular vīṭṭil: oblique stem vīṭṭ-)
Turkish  evlerimizden           ev-ler-imiz-den              4 morph  house-PL-1PL.POSS-ABL 'from our houses'
Hindi    laṛkiyõ                laṛk-iyõ                     2 morph  child-F.PL.OBL (one suffix = gender+number+case)
English  went                   go+PAST                      1 morph  suppletion: no segmentable boundary


A greedy longest-suffix stripper on transliterated Kannada nouns shows the idea behind rule-based morphological analysers (real analysers also handle sandhi, the sound changes at morpheme boundaries).

In [7]:
KN_SUFFIXES = {"alli": "LOC", "annu": "ACC", "inda": "INS/ABL", "ige": "DAT", "gaḷ": "PL", "gaḷu": "PL"}
def strip_suffixes(word, suffixes, min_stem=2):
    tags = []
    changed = True
    while changed:
        changed = False
        for suf in sorted(suffixes, key=len, reverse=True):
            if word.endswith(suf) and len(word) - len(suf) >= min_stem:
                tags.insert(0, (suf, suffixes[suf])); word = word[: -len(suf)]; changed = True; break
    return word, tags
for w in ["manegaḷalli", "maragaḷannu", "maneyinda", "manege", "pustakagaḷu"]:
    print(f"{w:14s} ->", strip_suffixes(w, KN_SUFFIXES))

manegaḷalli    -> ('mane', [('gaḷ', 'PL'), ('alli', 'LOC')])
maragaḷannu    -> ('mara', [('gaḷ', 'PL'), ('annu', 'ACC')])
maneyinda      -> ('maney', [('inda', 'INS/ABL')])
manege         -> ('manege', [])
pustakagaḷu    -> ('pustaka', [('gaḷu', 'PL')])


Note the failures that a real analyser must fix: *maneyinda* has an inserted glide *y* (sandhi), so the stripper returns stem *maney*; *manege* loses a vowel (*mane + ige → manege*), so it is not split at all. These are exactly the phonology–morphology interactions discussed in the note.

## Part C — Syntax: a CKY parser from scratch

CKY needs a grammar in **Chomsky normal form (CNF)**: every rule is either A → B C (two non-terminals) or A → w (one word). The chart cell `table[i][j]` holds every non-terminal that can derive words i..j−1. We also store back-pointers so we can count and rebuild trees.

In [8]:
CNF_RULES = '''
S  -> NP VP
VP -> V NP | VP PP
NP -> Det N | NP PP
PP -> P NP
NP -> 'I' | 'she' | 'fish' | 'forks'
V  -> 'saw' | 'eats'
Det -> 'the' | 'a'
N  -> 'man' | 'telescope' | 'park' | 'hill'
P  -> 'with' | 'in' | 'on'
'''
grammar = nltk.CFG.fromstring(CNF_RULES)
print("is CNF:", grammar.is_chomsky_normal_form())

binary = defaultdict(list)   # (B, C) -> [A]
lexical = defaultdict(list)  # word -> [A]
for prod in grammar.productions():
    rhs = prod.rhs()
    if len(rhs) == 2: binary[(rhs[0].symbol(), rhs[1].symbol())].append(prod.lhs().symbol())
    else: lexical[rhs[0]].append(prod.lhs().symbol())

def cky(words):
    n = len(words)
    table = [[defaultdict(int) for _ in range(n + 1)] for _ in range(n + 1)]   # table[i][j][A] = number of subtrees
    back = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n + 1)]
    for i, w in enumerate(words):
        for A in lexical[w]:
            table[i][i + 1][A] += 1; back[i][i + 1][A].append(w)
    for span in range(2, n + 1):
        for i in range(0, n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                for B, nb in list(table[i][k].items()):
                    for C, nc in list(table[k][j].items()):
                        for A in binary.get((B, C), []):
                            table[i][j][A] += nb * nc
                            back[i][j][A].append((k, B, C))
    return table, back

def show_chart(words, table):
    n = len(words)
    w = 16
    print(" " * 6 + "".join(f"{j:<{w}}" for j in range(1, n + 1)))
    for i in range(n):
        row = f"{i:<6}"
        for j in range(1, n + 1):
            cell = ",".join(f"{A}" + (f"×{c}" if c > 1 else "") for A, c in sorted(table[i][j].items())) if j > i else ""
            row += f"{(cell or ('-' if j > i else '')):<{w}}"
        print(row)

words = "she eats fish with forks".split()
table, back = cky(words)
show_chart(words, table)
print("\nnumber of S parses:", table[0][len(words)]["S"])

is CNF: True
      1               2               3               4               5               
0     NP              -               S               -               S×2             
1                     V               VP              -               VP×2            
2                                     NP              -               NP              
3                                                     P               PP              
4                                                                     NP              

number of S parses: 2


Rebuild the trees from back-pointers and check them against NLTK's chart parser.

In [9]:
def build(back, i, j, A):
    out = []
    for bp in back[i][j][A]:
        if isinstance(bp, str): out.append(nltk.Tree(A, [bp]))
        else:
            k, B, C = bp
            for lt in build(back, i, k, B):
                for rt in build(back, k, j, C):
                    out.append(nltk.Tree(A, [lt, rt]))
    return out

mine = build(back, 0, len(words), "S")
ref = list(nltk.ChartParser(grammar).parse(words))
print("CKY trees:", len(mine), " ChartParser trees:", len(ref))
print("same set of trees:", set(map(str, mine)) == set(map(str, ref)))
for t in mine: t.pretty_print()

CKY trees: 2  ChartParser trees: 2
same set of trees: True
          S                     
  ________|____                  
 |             VP               
 |    _________|___              
 |   |             NP           
 |   |     ________|____         
 |   |    |             PP      
 |   |    |         ____|____    
 NP  V    NP       P         NP 
 |   |    |        |         |   
she eats fish     with     forks

              S                     
  ____________|____                  
 |                 VP               
 |         ________|________         
 |        VP                PP      
 |    ____|___          ____|____    
 NP  V        NP       P         NP 
 |   |        |        |         |   
she eats     fish     with     forks



In [10]:
words2 = "I saw the man with the telescope".split()
t2, b2 = cky(words2)
show_chart(words2, t2)
print("\nparses:", t2[0][len(words2)]["S"])
for t in build(b2, 0, len(words2), "S"): print(t)

      1               2               3               4               5               6               7               
0     NP              -               -               S               -               -               S×2             
1                     V               -               VP              -               -               VP×2            
2                                     Det             NP              -               -               NP              
3                                                     N               -               -               -               
4                                                                     P               -               PP              
5                                                                                     Det             NP              
6                                                                                                     N               

parses: 2
(S
  (NP I)
  (VP
    (V saw)
    (NP

### Converting a non-CNF grammar

The slide grammar from the shared notebook has `VP -> V NP PP` (three children) and unit rules `NP -> N`. NLTK can binarise it; CKY then gives the same number of parses.

In [11]:
slide = nltk.CFG.fromstring('''
S  -> NP VP
VP -> V NP | V NP PP
NP -> N | NP PP
PP -> P NP
N  -> 'Alice' | 'strawberries' | 'chocolate'
V  -> 'eats'
P  -> 'with'
''')
print("slide grammar is CNF:", slide.is_chomsky_normal_form())
cnf = slide.chomsky_normal_form()
print("converted is CNF:", cnf.is_chomsky_normal_form())
for p in cnf.productions(): print("  ", p)
s = "Alice eats strawberries with chocolate".split()
print("ChartParser on original:", len(list(nltk.ChartParser(slide).parse(s))))
print("ChartParser on CNF     :", len(list(nltk.ChartParser(cnf).parse(s))))

slide grammar is CNF: False
converted is CNF: True
   VP -> V NP
   VP -> V VP@$@V
   NP -> NP PP
   N -> 'chocolate'
   NP -> 'chocolate'
   S -> NP VP
   P -> 'with'
   N -> 'Alice'
   NP -> 'Alice'
   V -> 'eats'
   VP@$@V -> NP PP
   PP -> P NP
   N -> 'strawberries'
   NP -> 'strawberries'
ChartParser on original: 2
ChartParser on CNF     : 2


### Why CKY is O(n³): count the inner-loop iterations

In [12]:
def triple_count(n):
    return sum(1 for span in range(2, n + 1) for i in range(n - span + 1) for k in range(i + 1, i + span))
for n in [5, 7, 10, 20, 40]:
    print(f"n={n:3d}  (i,k,j) triples = {triple_count(n):6d}   (n^3-n)/6 = {(n**3 - n)//6:6d}")

n=  5  (i,k,j) triples =     20   (n^3-n)/6 =     20
n=  7  (i,k,j) triples =     56   (n^3-n)/6 =     56
n= 10  (i,k,j) triples =    165   (n^3-n)/6 =    165
n= 20  (i,k,j) triples =   1330   (n^3-n)/6 =   1330
n= 40  (i,k,j) triples =  10660   (n^3-n)/6 =  10660


## Part D — Structural ambiguity explodes: parses vs number of PPs

Sentence pattern: *I saw the man* + *with the telescope* + *in the park* + *on the hill* + … Each extra PP can attach to the verb phrase or to any earlier noun phrase.

In [13]:
pps = ["with the telescope", "in the park", "on the hill", "with the man", "in the park", "on the hill"]
catalan = lambda m: math.comb(2 * m, m) // (m + 1)
for k in range(0, 6):
    sent = ("I saw the man " + " ".join(pps[:k])).split()
    t, _ = cky(sent)
    print(f"{k} PPs, {len(sent):2d} words -> {t[0][len(sent)]['S']:4d} parses   Catalan({k+1}) = {catalan(k+1)}")

0 PPs,  4 words ->    1 parses   Catalan(1) = 1
1 PPs,  7 words ->    2 parses   Catalan(2) = 2
2 PPs, 10 words ->    5 parses   Catalan(3) = 5
3 PPs, 13 words ->   14 parses   Catalan(4) = 14
4 PPs, 16 words ->   42 parses   Catalan(5) = 42
5 PPs, 19 words ->  132 parses   Catalan(6) = 132


## Part E — Lexical semantics with WordNet

In [14]:
def lemmas(s): return [l.name() for l in s.lemmas()]
car = wn.synset("car.n.01")
print("car.n.01:", car.definition())
print("  synonyms   :", lemmas(car))
print("  hypernyms  :", [h.name() for h in car.hypernyms()])
print("  part meronyms:", sorted(m.name() for m in car.part_meronyms())[:10], "...")
good = wn.synset("good.a.01")
print("good.a.01 antonyms:", [a.name() for l in good.lemmas() for a in l.antonyms()])
dog = wn.synset("dog.n.01")
path = dog.hypernym_paths()[0]
print("dog hypernym path:", " -> ".join(s.name().split('.')[0] for s in path))
tree = wn.synset("tree.n.01")
print("tree part meronyms:", sorted(m.name() for m in tree.part_meronyms()))
print("tree substance meronyms:", [m.name() for m in tree.substance_meronyms()])
print("hyponyms of 'vehicle.n.01' (first 8):", sorted(h.name() for h in wn.synset('vehicle.n.01').hyponyms())[:8])

car.n.01: a motor vehicle with four wheels; usually propelled by an internal combustion engine
  synonyms   : ['car', 'auto', 'automobile', 'machine', 'motorcar']
  hypernyms  : ['motor_vehicle.n.01']
  part meronyms: ['accelerator.n.01', 'air_bag.n.01', 'auto_accessory.n.01', 'automobile_engine.n.01', 'automobile_horn.n.01', 'buffer.n.06', 'bumper.n.02', 'car_door.n.01', 'car_mirror.n.01', 'car_seat.n.01'] ...
good.a.01 antonyms: ['bad']
dog hypernym path: entity -> physical_entity -> object -> whole -> living_thing -> organism -> animal -> chordate -> vertebrate -> mammal -> placental -> carnivore -> canine -> dog
tree part meronyms: ['burl.n.02', 'crown.n.07', 'limb.n.02', 'stump.n.01', 'trunk.n.01']
tree substance meronyms: ['heartwood.n.01', 'sapwood.n.01']
hyponyms of 'vehicle.n.01' (first 8): ['bumper_car.n.01', 'craft.n.02', 'military_vehicle.n.01', 'rocket.n.01', 'skibob.n.01', 'sled.n.01', 'steamroller.n.02', 'wheeled_vehicle.n.01']


In [15]:
pairs = [("dog.n.01", "cat.n.01"), ("dog.n.01", "car.n.01"), ("car.n.01", "bicycle.n.01"), ("dog.n.01", "dog.n.01")]
for a, b in pairs:
    A, B = wn.synset(a), wn.synset(b)
    print(f"{a:14s} {b:14s} path_sim = {A.path_similarity(B):.3f}  LCS = {A.lowest_common_hypernyms(B)[0].name()}")
for w in ["bank", "run", "set", "play", "fish", "time", "flies", "like"]:
    ss = wn.synsets(w)
    print(f"{w:6s} senses = {len(ss):3d}   by POS = {dict(Counter(s.pos() for s in ss))}")

dog.n.01       cat.n.01       path_sim = 0.200  LCS = carnivore.n.01
dog.n.01       car.n.01       path_sim = 0.077  LCS = whole.n.02
car.n.01       bicycle.n.01   path_sim = 0.200  LCS = wheeled_vehicle.n.01
dog.n.01       dog.n.01       path_sim = 1.000  LCS = dog.n.01
bank   senses =  18   by POS = {'n': 10, 'v': 8}
run    senses =  57   by POS = {'n': 16, 'v': 41}
set    senses =  45   by POS = {'n': 13, 'v': 25, 's': 7}
play   senses =  52   by POS = {'n': 17, 'v': 35}
fish   senses =   6   by POS = {'n': 4, 'v': 2}
time   senses =  15   by POS = {'n': 10, 'v': 5}
flies  senses =  20   by POS = {'n': 6, 'v': 14}
like   senses =  11   by POS = {'n': 2, 'v': 5, 'a': 3, 's': 1}


## Part F — Word sense disambiguation with the Lesk algorithm

**Simplified Lesk:** for each sense of the target word, count how many (non-stop) words its gloss **plus examples** share with the sentence; choose the sense with the largest overlap. First the two-sense hand example used in the note (glosses copied from WordNet `bank.n.01` and `bank.n.02`).

In [16]:
STOP = set(stopwords.words("english"))
def toks(text):
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP}

SENSES = {
    "bank#1 (river)": "sloping land (especially the slope beside a body of water). "
                      "they pulled the canoe up on the bank. he sat on the bank of the river and watched the currents",
    "bank#2 (money)": "a financial institution that accepts deposits and channels the money into lending activities. "
                      "he cashed a check at the bank. that bank holds the mortgage on my home",
}
sentences = ["I went to the bank to deposit money and pay my mortgage",
             "We sat on the bank of the river and watched the water"]
def simplified_lesk(sentence, senses, target="bank"):
    ctx = toks(sentence) - {target}
    scores = {}
    for name, sig in senses.items():
        overlap = ctx & (toks(sig) - {target})
        scores[name] = (len(overlap), sorted(overlap))
    best = max(scores, key=lambda k: scores[k][0])
    return best, scores
for s in sentences:
    best, scores = simplified_lesk(s, SENSES)
    print(s); print("  context words:", sorted(toks(s) - {'bank'}))
    for k, v in scores.items(): print(f"  {k:16s} overlap={v[0]} {v[1]}")
    print("  ->", best, "\n")

I went to the bank to deposit money and pay my mortgage
  context words: ['deposit', 'money', 'mortgage', 'pay', 'went']
  bank#1 (river)   overlap=0 []
  bank#2 (money)   overlap=2 ['money', 'mortgage']
  -> bank#2 (money) 

We sat on the bank of the river and watched the water
  context words: ['river', 'sat', 'watched', 'water']
  bank#1 (river)   overlap=4 ['river', 'sat', 'watched', 'water']
  bank#2 (money)   overlap=0 []
  -> bank#1 (river) 



Now the same idea over **all 18 WordNet senses**, compared with NLTK's built-in `nltk.wsd.lesk` (which uses the gloss only, without examples, and breaks ties by sense order).

In [17]:
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
def wn_simplified_lesk(sentence, word, pos=None):
    ctx = toks(sentence) - {word}
    best, best_score = None, -1
    for s in wn.synsets(word, pos=pos):
        sig = toks(s.definition() + " " + " ".join(s.examples())) - {word}
        sc = len(ctx & sig)
        if sc > best_score: best, best_score = s, sc
    return best, best_score
tests = sentences + ["The bank approved the loan for the new house",
                     "Fishermen stood on the muddy bank of the stream",
                     "You can bank on me to finish the job"]
for s in tests:
    mine, sc = wn_simplified_lesk(s, "bank")
    ref = lesk(word_tokenize(s), "bank")
    print(s)
    print(f"   simplified Lesk (gloss+examples): {mine.name():18s} score={sc}  {mine.definition()[:60]}")
    print(f"   nltk.wsd.lesk                   : {ref.name():18s}         {ref.definition()[:60]}\n")

I went to the bank to deposit money and pay my mortgage
   simplified Lesk (gloss+examples): depository_financial_institution.n.01 score=2  a financial institution that accepts deposits and channels t
   nltk.wsd.lesk                   : depository_financial_institution.n.01         a financial institution that accepts deposits and channels t

We sat on the bank of the river and watched the water
   simplified Lesk (gloss+examples): bank.n.01          score=4  sloping land (especially the slope beside a body of water)
   nltk.wsd.lesk                   : bank.n.01                  sloping land (especially the slope beside a body of water)

The bank approved the loan for the new house
   simplified Lesk (gloss+examples): bank.n.06          score=1  the funds held by a gambling house or the dealer in some gam
   nltk.wsd.lesk                   : bank.n.06                  the funds held by a gambling house or the dealer in some gam

Fishermen stood on the muddy bank of the stream
   simp

Lesk is a **weak baseline**: one-word overlaps decide everything, and ties are frequent. Look at the failures above: in *the bank approved the loan for the new house* both versions pick `bank.n.06` ("funds held by a gambling **house**") because the single shared word is *house*; *loan* does not appear in the financial gloss. With zero overlap (*muddy bank of the stream*: the gloss says "water", not "stream"), the answer is just the first sense in the loop. For the verb use *bank on me*, neither method is restricted to verbs, so both pick a noun sense. Modern WSD uses contextual embeddings (BERT-style) and reaches roughly 80% F1 on standard benchmarks, versus a Lesk-family baseline in the 50s–60s (see the note).

## Part G — First-order logic with `nltk.sem`

Translate sentences to FOL, then **evaluate** them in a tiny world (a *model*): a domain of individuals plus the extension of every predicate.

In [18]:
from nltk.sem import Valuation, Model, Assignment
read = nltk.sem.Expression.fromstring
fol = {
    "Alice eats strawberries":            r"eat(alice, strawberries)",
    "Every student likes NLP":            r"all x.(student(x) -> like(x, nlp))",
    "Some student likes NLP":             r"exists x.(student(x) & like(x, nlp))",
    "No student failed":                  r"-exists x.(student(x) & failed(x))",
    "Every student reads some book":      r"all x.(student(x) -> exists y.(book(y) & read(x, y)))",
    "Some book is read by every student": r"exists y.(book(y) & all x.(student(x) -> read(x, y)))",
}
val = Valuation.fromstring('''
alice => a
ravi => r
nlp => n
strawberries => s
student => {a, r}
book => {b1, b2}
failed => {}
like => {(a, n), (r, n)}
eat => {(a, s)}
read => {(a, b1), (r, b2)}
''')
dom = val.domain
m = Model(dom, val); g = Assignment(dom)
for eng, f in fol.items():
    e = read(f)
    print(f"{eng:36s} {str(e):60s} -> {m.evaluate(f, g)}")

Alice eats strawberries              eat(alice,strawberries)                                      -> True
Every student likes NLP              all x.(student(x) -> like(x,nlp))                            -> True
Some student likes NLP               exists x.(student(x) & like(x,nlp))                          -> True
No student failed                    -exists x.(student(x) & failed(x))                           -> True
Every student reads some book        all x.(student(x) -> exists y.(book(y) & read(x,y)))         -> True
Some book is read by every student   exists y.(book(y) & all x.(student(x) -> read(x,y)))         -> False


The last two sentences show **quantifier-scope ambiguity**: *every student reads some book* is true in this model (each student reads *a* book), but the wide-scope reading *one particular book is read by every student* is false.

## Part H — Ambiguity tour

In [19]:
amb = nltk.CFG.fromstring('''
S  -> NP VP
NP -> N | N N | Det N
VP -> V NP | V PP
PP -> P NP
N  -> 'time' | 'flies' | 'arrow'
V  -> 'flies' | 'like' | 'time'
P  -> 'like'
Det -> 'an'
''')
sent = "time flies like an arrow".split()
parses = list(nltk.ChartParser(amb).parse(sent))
print(len(parses), "parses of 'time flies like an arrow' under a toy grammar:")
for t in parses: print("  ", t)
print("\nNLTK tagger picks one:", nltk.pos_tag(sent))

2 parses of 'time flies like an arrow' under a toy grammar:
   (S (NP (N time) (N flies)) (VP (V like) (NP (Det an) (N arrow))))
   (S
  (NP (N time))
  (VP (V flies) (PP (P like) (NP (Det an) (N arrow)))))



NLTK tagger picks one: [('time', 'NN'), ('flies', 'NNS'), ('like', 'IN'), ('an', 'DT'), ('arrow', 'NN')]


Readings: (1) *time* passes quickly, as an arrow does (N V PP); (2) *time-flies* (a kind of fly) are fond of an arrow (N N V NP); (3) the imperative *time flies (measure them) the way you would time an arrow* is missing because the toy grammar has no imperative rule `S -> VP` — add it and count again.

In [20]:
amb2 = nltk.CFG.fromstring('''
S  -> NP VP | VP
NP -> N | N N | Det N
VP -> V NP | V PP | V NP PP
PP -> P NP
N  -> 'time' | 'flies' | 'arrow'
V  -> 'flies' | 'like' | 'time'
P  -> 'like'
Det -> 'an'
''')
parses = list(nltk.ChartParser(amb2).parse(sent))
print(len(parses), "parses with the imperative rule:")
for t in parses: print("  ", t)

3 parses with the imperative rule:
   (S (NP (N time) (N flies)) (VP (V like) (NP (Det an) (N arrow))))
   (S
  (NP (N time))
  (VP (V flies) (PP (P like) (NP (Det an) (N arrow)))))
   (S
  (VP (V time) (NP (N flies)) (PP (P like) (NP (Det an) (N arrow)))))


## Part I — Phonetics: formants from a tube, a synthetic vowel, and LPC

**Uniform-tube model.** A neutral vocal tract (schwa) behaves like a tube closed at the glottis and open at the lips. Its resonances (formants) are the odd quarter-wavelength frequencies

F_n = (2n − 1) · c / (4L),  with c ≈ 350 m/s (warm, moist air).

In [21]:
import numpy as np
from scipy.signal import lfilter
from scipy.linalg import solve_toeplitz

c = 350.0
for L_cm in [17.5, 15.0, 14.0]:
    L = L_cm / 100
    print(f"L = {L_cm:4.1f} cm  ->  F1, F2, F3 =", [round((2 * n - 1) * c / (4 * L), 1) for n in (1, 2, 3)], "Hz")

L = 17.5 cm  ->  F1, F2, F3 = [500.0, 1500.0, 2500.0] Hz
L = 15.0 cm  ->  F1, F2, F3 = [583.3, 1750.0, 2916.7] Hz
L = 14.0 cm  ->  F1, F2, F3 = [625.0, 1875.0, 3125.0] Hz


**Source–filter synthesis.** Speech = a source (glottal pulses at the pitch F0) passed through a filter (the vocal tract, whose resonances are the formants). Below: a 120 Hz pulse train through three second-order resonators set to Peterson–Barney average male formants for /i/, /ɑ/, /u/. Then **linear predictive coding (LPC)** recovers the formants from the waveform alone: fit an all-pole model by solving the Toeplitz normal equations, take the roots of the predictor polynomial, and convert root angles to Hz.

In [22]:
fs, F0 = 10000, 120
n = fs // 2
source = np.zeros(n); source[:: fs // F0] = 1.0           # glottal impulse train

def resonator(x, F, B):
    r, th = np.exp(-np.pi * B / fs), 2 * np.pi * F / fs
    return lfilter([1 - r], [1, -2 * r * np.cos(th), r * r], x)

def synth(formants, bws=(60, 90, 120)):
    y = source
    for F, B in zip(formants, bws): y = resonator(y, F, B)
    return y

def lpc_formants(x, order=12):
    x = lfilter([1, -0.97], [1], x) * np.hamming(len(x))  # pre-emphasis + window
    r = np.correlate(x, x, "full")[len(x) - 1 : len(x) + order]
    a = solve_toeplitz(r[:order], r[1 : order + 1])        # autocorrelation method
    roots = np.roots(np.r_[1, -a]); roots = roots[np.imag(roots) > 0]
    f = np.angle(roots) * fs / (2 * np.pi); bw = -fs / np.pi * np.log(np.abs(roots))
    return sorted(round(fi) for fi, b in zip(f, bw) if fi > 150 and b < 400)

VOWELS = {"/i/ beet": (270, 2290, 3010), "/ɑ/ father": (730, 1090, 2440), "/u/ boot": (300, 870, 2240)}
for v, F in VOWELS.items():
    est = lpc_formants(synth(F)[2000:2512])[:3]
    print(f"{v:11s} true {F}  LPC estimate {tuple(est)}")

/i/ beet    true (270, 2290, 3010)  LPC estimate (308, 2290, 3009)
/ɑ/ father  true (730, 1090, 2440)  LPC estimate (730, 1087, 2439)
/u/ boot    true (300, 870, 2240)  LPC estimate (371, 872, 2247)


F2 and F3 are recovered within a few Hz. F1 of the close vowels /i/ and /u/ (270–300 Hz) is overestimated by tens of Hz because a 120 Hz source only samples the spectrum at harmonics 120, 240, 360 … Hz, so a low resonance is poorly pinned down — a real limitation of formant tracking for high-pitched voices. The **F1–F2 plane** separates the vowels: F1 tracks tongue **height** (low F1 = close vowel), F2 tracks **backness** (high F2 = front vowel). See `figures_01.py` for spectrograms.

## Part J — Phonology: minimal pairs and a general rewrite-rule engine

A **minimal pair** is two words whose pronunciations differ in exactly one phoneme. Finding one proves the two sounds are separate phonemes. We mine CMUdict (restricted to words that WordNet also knows, to avoid surnames).

In [23]:
real = {w for w in cmu if w.isalpha() and wn.synsets(w)}
by_pron = defaultdict(set)
for w in real:
    for p in cmu[w]: by_pron[tuple(strip(p))].add(w)

def minimal_pairs(x, y):
    found = set()
    for key, ws in by_pron.items():
        for i, ph in enumerate(key):
            if ph == x:
                other = key[:i] + (y,) + key[i + 1 :]
                for a in ws:
                    for b in by_pron.get(other, ()):
                        if a != b: found.add((a, b))
    return found

print("dictionary words checked:", len(real))
for x, y in [("R", "L"), ("P", "B"), ("T", "D"), ("S", "Z"), ("IY", "IH"), ("TH", "DH")]:
    mp = minimal_pairs(x, y)
    show = [p for p in sorted(mp) if p[0] in {"right", "pray", "fry", "pat", "pin", "tie", "sip", "sheep", "beat", "ether", "teeth"}]
    print(f"{x:2s}/{y:2s}: {len(mp):5d} pairs   e.g. {show[:3] or sorted(mp)[:3]}")

dictionary words checked: 51434
R /L :  1123 pairs   e.g. [('fry', 'fly'), ('pray', 'play'), ('right', 'light')]
P /B :   622 pairs   e.g. [('pat', 'bat'), ('pin', 'been'), ('pin', 'bin')]
T /D :   868 pairs   e.g. [('beat', 'bead'), ('beat', 'bede'), ('pat', 'pad')]
S /Z :   321 pairs   e.g. [('sip', 'zip')]
IY/IH:   434 pairs   e.g. [('beat', 'bit'), ('sheep', 'ship')]
TH/DH:     5 pairs   e.g. [('ether', 'either'), ('teeth', 'teethe')]


/θ/ vs /ð/ (*ether/either*, *teeth/teethe*) has only a handful of minimal pairs: the contrast is phonemic but has a **low functional load**. /r/ vs /l/ has over a thousand, which is why the Japanese-speaker difficulty in the slides matters so much for intelligibility.

**A general rule engine.** A phonological rule A → B / C _ D rewrites A as B when it is preceded by something in C and followed by something in D. Example: English **nasal place assimilation** of the prefix *in-*: N → M / _ {P, B, M} and N → NG / _ {K, G}.

In [24]:
def apply_rule(seq, A, B, left=None, right=None):
    out = list(seq)
    for i, s in enumerate(seq):
        if s != A: continue
        if left is not None and (i == 0 or seq[i - 1] not in left): continue
        if right is not None and (i == len(seq) - 1 or seq[i + 1] not in right): continue
        out[i] = B
    return out

LABIAL, VELAR = {"P", "B", "M"}, {"K", "G"}
for stem in ["possible", "balance", "mature", "correct", "complete", "definite", "tolerant", "visible"]:
    underlying = ["IH", "N"] + strip(cmu[stem][0])
    surface = apply_rule(apply_rule(underlying, "N", "M", right=LABIAL), "N", "NG", right=VELAR)
    print(f"in+{stem:9s} {' '.join(underlying):26s} -> {' '.join(surface):26s} CMU: {' '.join(strip(cmu['in' + stem if stem[0] not in 'pbm' else 'im' + stem][0]))}")

in+possible  IH N P AA S AH B AH L      -> IH M P AA S AH B AH L      CMU: IH M P AA S AH B AH L
in+balance   IH N B AE L AH N S         -> IH M B AE L AH N S         CMU: IH M B AE L AH N S
in+mature    IH N M AH CH UH R          -> IH M M AH CH UH R          CMU: IH M AH T Y UH R
in+correct   IH N K ER EH K T           -> IH NG K ER EH K T          CMU: IH N K ER EH K T
in+complete  IH N K AH M P L IY T       -> IH NG K AH M P L IY T      CMU: IH N K AH M P L IY T
in+definite  IH N D EH F AH N AH T      -> IH N D EH F AH N AH T      CMU: IH N D EH F AH N AH T
in+tolerant  IH N T AA L ER AH N T      -> IH N T AA L ER AH N T      CMU: IH N T AA L ER AH N T
in+visible   IH N V IH Z AH B AH L      -> IH N V IH Z AH B AH L      CMU: IH N V IH Z AH B AH L


The rule predicts /m/ before labials and [ŋ] before velars. CMUdict agrees on the labial cases (spelling also changes: *im*possible) but writes N in *incorrect*: the dictionary is a **phonemic** transcription, and [ŋ] before /k/ is a predictable **allophone** of /n/ that a phonemic dictionary does not record. *immature* also shows degemination (two M's surface as one).

## Part K — Pragmatics and discourse

### K1. A rule-based speech-act tagger (Searle's five classes, plus questions)

In [25]:
SA_RULES = [
    (r"^(can|could|would|will) you\b", "DIRECTIVE (indirect request)"),
    (r"^(please\b|turn|book|play|set|call|open|close|pass)\b", "DIRECTIVE (command)"),
    (r"\b(i promise|i will|i'll)\b", "COMMISSIVE"),
    (r"\b(thank|sorry|congratulations|apolog)", "EXPRESSIVE"),
    (r"\b(i now pronounce|you're fired|i resign|hereby)\b", "DECLARATION"),
    (r"\?$", "QUESTION"),
]
def speech_act(u):
    s = u.lower().strip()
    for pat, label in SA_RULES:
        if re.search(pat, s): return label
    return "ASSERTIVE"

for u in ["Can you pass the salt?", "Turn off the lights.", "I promise I'll call tomorrow.", "Thanks a lot for the help!",
          "I hereby declare the meeting open.", "What time is it?", "It's cold in here.", "Could you open the window?"]:
    print(f"{u:38s} -> {speech_act(u)}")

Can you pass the salt?                 -> DIRECTIVE (indirect request)
Turn off the lights.                   -> DIRECTIVE (command)
I promise I'll call tomorrow.          -> COMMISSIVE
Thanks a lot for the help!             -> EXPRESSIVE
I hereby declare the meeting open.     -> DECLARATION
What time is it?                       -> QUESTION
It's cold in here.                     -> ASSERTIVE
Could you open the window?             -> DIRECTIVE (indirect request)


Surface patterns catch conventional indirect requests (*can you …*), but *It's cold in here* comes out ASSERTIVE although in context it may be a request to close the window. Inferring that needs the situation, not the string — the core reason pragmatics is hard.

### K2. Pronoun resolution: agreement + recency + subject preference + a binding constraint

In [26]:
ENT = {"John": ("m", "sg"), "Mary": ("f", "sg"), "Susan": ("f", "sg"), "car": ("n", "sg"), "book": ("n", "sg"),
       "keys": ("n", "pl"), "Ravi": ("m", "sg"), "laptop": ("n", "sg")}
PRON = {"he": ("m", "sg"), "him": ("m", "sg"), "she": ("f", "sg"), "her": ("f", "sg"), "it": ("n", "sg"), "they": ("n", "pl")}
ROLE_RANK = {"S": 0, "O": 1, "X": 2}

def resolve(discourse):
    history, out = [], []
    for si, sent in enumerate(discourse):
        used = set()                                   # binding: two pronouns in one clause are not co-referent
        for w, role in sent:
            if w.lower() in PRON:
                cands = [(e, r, sj) for e, r, sj in history if ENT[e] == PRON[w.lower()] and e not in used]
                cands.sort(key=lambda t: (-t[2], ROLE_RANK[t[1]]))   # most recent sentence first, subject first
                ans = cands[0][0] if cands else None
                used.add(ans); out.append((w, ans))
        history += [(w, role, si) for w, role in sent if w in ENT]
    return out

print(resolve([[("John", "S"), ("car", "O")], [("He", "S"), ("it", "O")]]))
print(resolve([[("Mary", "S"), ("Susan", "O")], [("She", "S"), ("her", "O"), ("book", "O")]]))
print(resolve([[("Ravi", "S"), ("laptop", "O"), ("keys", "X")], [("They", "S")], [("It", "S")]]))

[('He', 'John'), ('it', 'car')]
[('She', 'Mary'), ('her', 'Susan')]
[('They', 'keys'), ('It', 'laptop')]


### K3. Centering theory: transitions for a four-utterance discourse

For each utterance we list its forward-looking centres Cf in salience order (subject > object > other). The backward-looking centre Cb(Uₙ) is the highest-ranked element of Cf(Uₙ₋₁) that is realised in Uₙ; the preferred centre Cp(Uₙ) is the first element of Cf(Uₙ).

In [27]:
def centering(cfs):
    rows, prev_cb = [], None
    for i, cf in enumerate(cfs):
        cp = cf[0]
        if i == 0:
            rows.append((cf, None, cp, "-")); continue
        cb = next((e for e in cfs[i - 1] if e in cf), None)
        same = (prev_cb is None) or (cb == prev_cb)
        t = ("CONTINUE" if cb == cp else "RETAIN") if same else ("SMOOTH-SHIFT" if cb == cp else "ROUGH-SHIFT")
        rows.append((cf, cb, cp, t)); prev_cb = cb
    return rows

utts = ["John went to his favourite music store to buy a piano.",
        "He had frequented the store for many years.",
        "He was excited that he could finally buy a piano.",
        "It was closing just as John arrived."]
cfs = [["John", "store", "piano"], ["John", "store"], ["John", "piano"], ["store", "John"]]
for u, (cf, cb, cp, t) in zip(utts, centering(cfs)):
    print(f"{u:55s} Cf={cf}  Cb={cb}  Cp={cp}  -> {t}")

John went to his favourite music store to buy a piano.  Cf=['John', 'store', 'piano']  Cb=None  Cp=John  -> -
He had frequented the store for many years.             Cf=['John', 'store']  Cb=John  Cp=John  -> CONTINUE
He was excited that he could finally buy a piano.       Cf=['John', 'piano']  Cb=John  Cp=John  -> CONTINUE
It was closing just as John arrived.                    Cf=['store', 'John']  Cb=John  Cp=store  -> RETAIN


U4 is a RETAIN: John is still the Cb but the store has become the most salient entity, and readers find this discourse slightly jarring at *It*. Centering predicts the preference order CONTINUE > RETAIN > SMOOTH-SHIFT > ROUGH-SHIFT.

## Part L — Checks for the note's practice problems

In [28]:
# P-numbers refer to the practice bank in the note
print("P4 tube L=15 cm:", [round((2*k-1)*350/(4*0.15), 1) for k in (1, 2, 3)])

pl_words = ["map", "bed", "wish", "laugh", "month", "judge"]
pa_words = ["hope", "beg", "wait", "buzz"]
print("P5 plural:", {w: "/" + "".join(plural_suffix(cmu[w][0])) + "/" for w in pl_words})
print("P5 past  :", {w: "/" + "".join(past_suffix(cmu[w][0])) + "/" for w in pa_words})

for s in ["she saw the man in the park", "I saw the man on the hill with the telescope"]:
    ws = s.split(); t, b = cky(ws)
    print(f"P12/P14 '{s}': n={len(ws)}, S parses = {t[0][len(ws)]['S']}")
ws = "she saw the man in the park".split(); t, _ = cky(ws); show_chart(ws, t)

for n in [6, 10, 20]:
    print(f"P13 n={n}: chart cells n(n+1)/2 = {n*(n+1)//2}, split triples = {triple_count(n)}")
print("P13 ratio of triples n=20 vs n=10:", round(triple_count(20) / triple_count(10), 2))
print("P14 Catalan(4), Catalan(5), Catalan(7):", catalan(4), catalan(5), catalan(7))

for a, b in [("dog.n.01", "cat.n.01"), ("car.n.01", "bicycle.n.01"), ("dog.n.01", "wolf.n.01"), ("car.n.01", "truck.n.01")]:
    A, B = wn.synset(a), wn.synset(b)
    print(f"P18 path_sim({a}, {b}) = {A.path_similarity(B):.3f}, edges = {A.shortest_path_distance(B)}")

for s in ["We dragged the boat out of the water onto the bank", "The bank will not lend money without a mortgage"]:
    best, scores = simplified_lesk(s, SENSES)
    print("P19", s, "->", best, {k: v for k, v in scores.items()})

seg_counts = [len(ms) for _, _, ms, _ in segmentations]
print("P10 index of synthesis (morphemes/word) for the 8 Part-B words:", round(sum(seg_counts) / len(seg_counts), 3))

P4 tube L=15 cm: [583.3, 1750.0, 2916.7]
P5 plural: {'map': '/S/', 'bed': '/Z/', 'wish': '/IHZ/', 'laugh': '/S/', 'month': '/S/', 'judge': '/IHZ/'}
P5 past  : {'hope': '/T/', 'beg': '/D/', 'wait': '/IHD/', 'buzz': '/D/'}
P12/P14 'she saw the man in the park': n=7, S parses = 2
P12/P14 'I saw the man on the hill with the telescope': n=10, S parses = 5
      1               2               3               4               5               6               7               
0     NP              -               -               S               -               -               S×2             
1                     V               -               VP              -               -               VP×2            
2                                     Det             NP              -               -               NP              
3                                                     N               -               -               -               
4                                                   

## Part M — More practice-bank checks (speech acts, centering, CKY worked example)

Checks for P17, P20 and P22 of the note's practice bank, plus a cell-by-cell trace of the CKY worked example in §9 (which split point filled each cell).

In [29]:
# P20: speech-act tagger on new utterances
for u in ["Could you send the report by 5?", "I promise to be there at 6.", "Sorry for the delay.",
          "The meeting is at noon.", "I hereby resign.", "Is the shop open?"]:
    print(f"{u:36s} -> {speech_act(u)}")

# P22: centering for a four-utterance discourse
utts22 = ["Ravi met Kiran at the station.", "He gave him a ticket.", "Kiran thanked him.", "Kiran boarded the train."]
cfs22 = [["Ravi", "Kiran", "station"], ["Ravi", "Kiran", "ticket"], ["Kiran", "Ravi"], ["Kiran", "train"]]
for u, (cf, cb, cp, t) in zip(utts22, centering(cfs22)):
    print(f"{u:32s} Cf={cf}  Cb={cb}  Cp={cp}  -> {t}")

# P17: evaluate two more FOL formulas in the Part G model
for f in [r"all x.(student(x) -> exists y.(book(y) & read(x, y)))", r"exists x.(student(x) & failed(x))",
          r"all x.(book(x) -> exists y.(student(y) & read(y, x)))"]:
    print(f"{f:60s} -> {m.evaluate(f, g)}")

Could you send the report by 5?      -> DIRECTIVE (indirect request)
I promise to be there at 6.          -> COMMISSIVE
Sorry for the delay.                 -> EXPRESSIVE
The meeting is at noon.              -> ASSERTIVE
I hereby resign.                     -> DECLARATION
Is the shop open?                    -> QUESTION
Ravi met Kiran at the station.   Cf=['Ravi', 'Kiran', 'station']  Cb=None  Cp=Ravi  -> -
He gave him a ticket.            Cf=['Ravi', 'Kiran', 'ticket']  Cb=Ravi  Cp=Ravi  -> CONTINUE
Kiran thanked him.               Cf=['Kiran', 'Ravi']  Cb=Ravi  Cp=Kiran  -> RETAIN
Kiran boarded the train.         Cf=['Kiran', 'train']  Cb=Kiran  Cp=Kiran  -> SMOOTH-SHIFT
all x.(student(x) -> exists y.(book(y) & read(x, y)))        -> True
exists x.(student(x) & failed(x))                            -> False
all x.(book(x) -> exists y.(student(y) & read(y, x)))        -> True


In [30]:
# Trace of the CKY worked example: which (B, C, split) produced each entry
words = "she eats fish with forks".split()
table, back = cky(words)
n = len(words)
for span in range(1, n + 1):
    for i in range(n - span + 1):
        j = i + span
        if table[i][j]:
            for A, bps in back[i][j].items():
                print(f"[{i},{j}] '{' '.join(words[i:j])}': {A} <- {bps}")

[0,1] 'she': NP <- ['she']
[1,2] 'eats': V <- ['eats']
[2,3] 'fish': NP <- ['fish']
[3,4] 'with': P <- ['with']
[4,5] 'forks': NP <- ['forks']
[1,3] 'eats fish': VP <- [(2, 'V', 'NP')]
[3,5] 'with forks': PP <- [(4, 'P', 'NP')]
[0,3] 'she eats fish': S <- [(1, 'NP', 'VP')]
[2,5] 'fish with forks': NP <- [(3, 'NP', 'PP')]
[1,5] 'eats fish with forks': VP <- [(2, 'V', 'NP'), (3, 'VP', 'PP')]
[0,5] 'she eats fish with forks': S <- [(1, 'NP', 'VP')]
